# Fundamentos de Agentes: Teoría y Mecánica Interna

## Objetivo

Entender los agentes desde los **First Principles** (principios fundamentales), sin abstracciones. Aprenderás la mecánica interna, los riesgos y cómo funcionan realmente antes de usar librerías como LangChain.

### ¿Por qué este enfoque?

Muchos tutoriales saltan directamente a usar `AgentExecutor` sin explicar:
- ¿Qué es realmente un agente?
- ¿Cómo funciona internamente?
- ¿Qué riesgos tiene?
- ¿Cómo se previenen los bucles infinitos?

En este notebook entenderás **cómo construir un agente desde cero** antes de usar las herramientas de LangChain.


In [ ]:
# Setup inicial
import sys
import os
import re

# Hack para importar desde src
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.models import get_local_llm

print("✓ Imports completados")


## 1. El Cambio de Paradigma: Chain vs. Loop

### Chains: DAG Predecible

Una **Chain** (cadena) en LangChain es un **DAG (Grafo Acíclico Dirigido)** predecible:

```
Input → Componente A → Componente B → Componente C → Output
```

**Características:**
- ✅ **Predecible**: Siempre sigue el mismo camino
- ✅ **Determinista**: Mismo input = mismo output
- ✅ **Estático**: La estructura no cambia durante la ejecución
- ✅ **Sin retroalimentación**: No puede "volver atrás"

**Ejemplo de Chain:**
```python
chain = prompt | llm | parser
# Siempre: prompt → llm → parser (en ese orden)
```

### Agentes: Bucle Gobernado por LLM

Un **Agente** es un **bucle (`while`)** gobernado por un LLM:

```
while not finished:
    thought = llm.think(history)
    if thought == "use_tool":
        result = execute_tool()
        history.append(result)
    elif thought == "final_answer":
        break
```

**Características:**
- 🔄 **Dinámico**: El camino puede cambiar según la situación
- 🧠 **Razonamiento**: El LLM decide qué hacer en cada paso
- 🔀 **No determinista**: Mismo input puede llevar a diferentes caminos
- 🔁 **Con retroalimentación**: Puede usar resultados previos para decidir

### El Concepto "Hands-off" (Autonomía)

Un agente es **"hands-off"** (autónomo):
- El sistema opera **solo** hasta que decide parar
- No necesitas intervenir en cada paso
- El LLM toma decisiones sobre:
  - Qué herramienta usar
  - Cuándo parar
  - Cómo combinar resultados

**Riesgo:** Sin controles adecuados, un agente puede:
- Entrar en bucles infinitos
- Usar herramientas peligrosas
- Consumir recursos indefinidamente


## 2. Arquitectura Cognitiva: ReAct Pattern

### ¿Qué es ReAct?

**ReAct** = **Reason** (Razonar) + **Act** (Actuar)

Es un patrón donde el agente alterna entre:
1. **Pensar** (Reason): Analizar la situación
2. **Actuar** (Act): Ejecutar una herramienta
3. **Observar** (Observe): Ver el resultado
4. **Repetir** hasta tener la respuesta final

### El Ciclo ReAct Completo

```
┌─────────────────────────────────────────────────┐
│  Input: "¿Cuánto es 5 por 5?"                   │
└─────────────────────────────────────────────────┘
                    ↓
┌─────────────────────────────────────────────────┐
│  THOUGHT: Necesito multiplicar 5 por 5          │
│  ACTION: multiply                               │
│  ACTION_INPUT: {"a": 5, "b": 5}                 │
└─────────────────────────────────────────────────┘
                    ↓
┌─────────────────────────────────────────────────┐
│  OBSERVATION: 25                                 │
└─────────────────────────────────────────────────┘
                    ↓
┌─────────────────────────────────────────────────┐
│  THOUGHT: Tengo la respuesta                     │
│  FINAL_ANSWER: 5 por 5 es 25                    │
└─────────────────────────────────────────────────┘
```

### Desglose del Ciclo

1. **Input**: La pregunta del usuario
2. **Thought (Razonar)**: El LLM analiza qué necesita hacer
3. **Action (Decidir Tool)**: Elige qué herramienta usar
4. **Action Input (Parámetros)**: Los argumentos para la herramienta
5. **Observation (Resultado real)**: El resultado de ejecutar la herramienta
6. **Repeat**: Vuelve al paso 2 si necesita más información
7. **Final Answer**: Cuando tiene suficiente información, genera la respuesta final


## 3. El Componente Invisible: "The Scratchpad"

### ¿Qué es el Scratchpad?

El **Scratchpad** (bloc de notas) es un **string que crece** con cada iteración del agente. Contiene todo el historial de pensamientos, acciones y observaciones.

**CRÍTICO:** El agente **NO tiene memoria real**. Usa el scratchpad como su "memoria de trabajo".

### ¿Por qué es Necesario?

Sin scratchpad, el agente:
- ❌ No recordaría qué herramientas ya intentó usar
- ❌ No sabría qué resultados obtuvo
- ❌ Entraría en bucles infinitos repitiendo lo mismo
- ❌ No podría aprender de errores previos

### Estructura del Scratchpad

```
Thought: Necesito buscar información sobre el clima
Action: get_weather
Action Input: {"city": "Madrid"}
Observation: La temperatura en Madrid es 22°C

Thought: Ahora necesito multiplicar esa temperatura por 2
Action: multiply
Action Input: {"a": 22, "b": 2}
Observation: 44

Thought: Tengo toda la información necesaria
Final Answer: La temperatura en Madrid es 22°C, y multiplicada por 2 es 44.
```

### El Scratchpad en Cada Iteración

En cada paso del bucle:
1. El scratchpad se envía al LLM como contexto
2. El LLM ve todo lo que ha pasado antes
3. El LLM decide el siguiente paso basándose en el historial
4. El nuevo paso se añade al scratchpad
5. El ciclo se repite

**Sin scratchpad = Agente sin memoria = Bucle infinito**


## 4. Seguridad y Control: Guardrails

### ¿Qué son los Guardrails?

**Guardrails** (barreras de seguridad) son mecanismos de defensa para agentes autónomos. Previenen comportamientos peligrosos o costosos.

### Los 3 Guardrails Esenciales

#### 1. Max Iterations (Máximo de Iteraciones)

**Problema:** Un agente puede entrar en un bucle infinito.

**Solución:** Limitar el número máximo de iteraciones.

```python
max_iterations = 5
iteration_count = 0

while iteration_count < max_iterations:
    # ... lógica del agente ...
    iteration_count += 1
```

**Riesgo sin este guardrail:**
- ⚠️ Coste infinito (cada iteración consume tokens)
- ⚠️ Tiempo infinito (puede ejecutarse para siempre)
- ⚠️ Recursos agotados

#### 2. Tool Whitelisting (Lista Blanca de Herramientas)

**Problema:** Un agente podría intentar usar herramientas peligrosas.

**Solución:** Solo permitir herramientas específicas y seguras.

```python
ALLOWED_TOOLS = ["multiply", "get_weather", "calc_length"]

if tool_name not in ALLOWED_TOOLS:
    raise SecurityError(f"Tool {tool_name} not allowed")
```

**Riesgo sin este guardrail:**
- ⚠️ Ejecución de código arbitrario (`os.system`, `eval`)
- ⚠️ Acceso a archivos del sistema
- ⚠️ Llamadas a APIs externas no autorizadas

#### 3. Parser Check (Validación de Parsing)

**Problema:** El LLM puede "alucinar" y devolver formato incorrecto.

**Solución:** Validar que la respuesta del LLM tenga el formato esperado.

```python
def parse_llm_response(response):
    # Buscar patrón "ACTION: tool_name"
    match = re.search(r"ACTION: (\w+)", response)
    if not match:
        raise ParsingError("Invalid format")
    return match.group(1)
```

**Riesgo sin este guardrail:**
- ⚠️ El agente puede devolver texto sin formato
- ⚠️ No se puede extraer la acción a ejecutar
- ⚠️ El bucle puede romperse o comportarse inesperadamente

### Resumen de Guardrails

| Guardrail | Protege Contra | Implementación |
|-----------|---------------|----------------|
| Max Iterations | Bucles infinitos | Contador en el bucle |
| Tool Whitelisting | Herramientas peligrosas | Lista de herramientas permitidas |
| Parser Check | Formato incorrecto | Validación y parsing robusto |


## 5. Simulación Manual: El "Ah-ha!" Moment

Ahora vamos a construir un agente ReAct **desde cero**, sin usar las clases de LangChain. Esto te mostrará que un agente es simplemente un `while` loop con parsing de strings.

### La Herramienta Dummy

Primero, definamos una herramienta simple:


In [ ]:
# Herramienta dummy: calcular la longitud de un texto
def calc_length(text: str) -> int:
    """
    Calcula la longitud de un texto.
    
    Args:
        text: El texto del cual calcular la longitud
        
    Returns:
        La longitud del texto en caracteres
    """
    return len(text)

# Probar la herramienta
print(f"calc_length('Hola') = {calc_length('Hola')}")
print(f"calc_length('Python es genial') = {calc_length('Python es genial')}")


### El Agente Manual: Bucle While con Parsing

Ahora construimos el agente. Observa cómo es simplemente un bucle `while` con `if/else` y parsing de strings:


In [ ]:
from langchain_core.messages import HumanMessage

# Instanciar el modelo
llm = get_local_llm()

def run_manual_agent(question: str, max_iterations: int = 5):
    """
    Simula un agente ReAct manualmente, sin usar clases de LangChain.
    
    Esto demuestra que un agente es simplemente:
    - Un bucle while
    - Parsing de strings
    - Ejecución condicional de herramientas
    """
    
    # GUARDRAIL 1: Max iterations
    iteration = 0
    
    # El SCRATCHPAD: string que crece con cada iteración
    scratchpad = f"""Pregunta: {question}

"""
    
    print("=" * 60)
    print("AGENTE MANUAL - SIMULACIÓN PASO A PASO")
    print("=" * 60)
    print(f"Pregunta inicial: {question}\n")
    
    while iteration < max_iterations:
        iteration += 1
        print(f"[ITERACIÓN {iteration}]")
        print("-" * 60)
        
        # Construir el prompt con el scratchpad
        prompt = f"""Eres un agente que puede usar herramientas para responder preguntas.

Herramientas disponibles:
- calc_length(text): Calcula la longitud de un texto

Formato de respuesta:
- Si necesitas usar una herramienta, responde: "ACTION: calc_length\nINPUT: <texto>"
- Si tienes la respuesta final, responde: "FINAL_ANSWER: <tu respuesta>"

Historial (scratchpad):
{scratchpad}

¿Cuál es tu siguiente paso?"""
        
        # Enviar al LLM
        response = llm.invoke([HumanMessage(content=prompt)])
        llm_output = response.content.strip()
        
        print(f"Respuesta del LLM:\n{llm_output}\n")
        
        # GUARDRAIL 3: Parser Check
        # Buscar si el LLM quiere usar una herramienta
        action_match = re.search(r"ACTION:\s*(\w+)", llm_output, re.IGNORECASE)
        input_match = re.search(r"INPUT:\s*(.+)", llm_output, re.IGNORECASE)
        final_answer_match = re.search(r"FINAL_ANSWER:\s*(.+)", llm_output, re.IGNORECASE)
        
        # Si el LLM quiere dar la respuesta final
        if final_answer_match:
            answer = final_answer_match.group(1).strip()
            print("=" * 60)
            print("✓ AGENTE COMPLETADO")
            print("=" * 60)
            print(f"Respuesta final: {answer}")
            return answer
        
        # Si el LLM quiere usar una herramienta
        if action_match and input_match:
            tool_name = action_match.group(1).strip()
            tool_input = input_match.group(1).strip()
            
            # GUARDRAIL 2: Tool Whitelisting
            if tool_name == "calc_length":
                print(f"→ Ejecutando herramienta: {tool_name}({tool_input})")
                
                # Ejecutar la herramienta
                try:
                    result = calc_length(tool_input)
                    observation = f"Resultado: {result}"
                    print(f"✓ {observation}\n")
                except Exception as e:
                    observation = f"Error: {str(e)}"
                    print(f"✗ {observation}\n")
                
                # Actualizar el scratchpad
                scratchpad += f"""Thought: {llm_output}
Action: {tool_name}
Action Input: {tool_input}
Observation: {observation}

"""
            else:
                # Herramienta no permitida
                observation = f"Error: Herramienta '{tool_name}' no está permitida"
                print(f"✗ {observation}\n")
                scratchpad += f"""Thought: {llm_output}
Observation: {observation}

"""
        else:
            # Formato incorrecto - el LLM no siguió el formato esperado
            print("⚠ Formato de respuesta inválido. Reintentando...\n")
            scratchpad += f"""Thought: {llm_output}
Observation: Formato inválido. Por favor, usa el formato correcto.

"""
    
    # Si llegamos aquí, se agotaron las iteraciones (GUARDRAIL 1 activado)
    print("=" * 60)
    print("✗ MÁXIMO DE ITERACIONES ALCANZADO")
    print("=" * 60)
    print("El agente no pudo completar la tarea en el límite de iteraciones.")
    return None


### Probar el Agente Manual

Ahora probemos el agente con diferentes preguntas:


In [ ]:
# Prueba 1: Pregunta simple
print("\n" + "=" * 60)
print("PRUEBA 1")
print("=" * 60)
resultado1 = run_manual_agent("¿Cuál es la longitud del texto 'Hola Mundo'?")


In [ ]:
# Prueba 2: Pregunta más compleja
print("\n" + "=" * 60)
print("PRUEBA 2")
print("=" * 60)
resultado2 = run_manual_agent("Calcula la longitud de 'Python' y luego dime si es mayor que 5")


### Análisis del Código: ¿Qué Hemos Aprendido?

Observa el código del agente manual. Es simplemente:

1. **Un bucle `while`** con límite (Guardrail 1)
2. **Parsing de strings** para extraer acciones (Guardrail 3)
3. **Validación de herramientas** antes de ejecutar (Guardrail 2)
4. **Un string que crece** (scratchpad) que actúa como memoria
5. **Condicionales `if/else`** para decidir qué hacer

**Esto es TODO lo que hace un agente.** Las librerías como LangChain solo abstraen esto, pero la mecánica fundamental es la misma.

### Diagrama del Flujo

```
┌─────────────────────────────────────────┐
│  Inicializar: iteration=0, scratchpad=""│
└─────────────────────────────────────────┘
              ↓
┌─────────────────────────────────────────┐
│  while iteration < max_iterations:    │
│    iteration += 1                       │
│    prompt = build_prompt(scratchpad)    │
│    response = llm.invoke(prompt)        │
│                                         │
│    if "FINAL_ANSWER" in response:       │
│      return answer                      │
│    elif "ACTION" in response:           │
│      if tool in ALLOWED_TOOLS:          │
│        result = execute_tool()          │
│        scratchpad += result             │
│      else:                              │
│        scratchpad += "Error: not allowed"│
│    else:                                │
│      scratchpad += "Invalid format"     │
└─────────────────────────────────────────┘
```

**Esto es un agente. Nada más.**


## Resumen: Conceptos Clave

### 1. Chain vs. Agent

| Aspecto | Chain | Agent |
|---------|-------|-------|
| Estructura | DAG predecible | Bucle dinámico |
| Control | Estático | Gobernado por LLM |
| Retroalimentación | No | Sí (scratchpad) |
| Autonomía | Baja | Alta (hands-off) |

### 2. ReAct Pattern

- **Reason**: El LLM razona sobre qué hacer
- **Act**: Ejecuta una herramienta
- **Observe**: Ve el resultado
- **Repeat**: Hasta tener la respuesta final

### 3. Scratchpad (Crítico)

- String que crece con cada iteración
- Contiene todo el historial de pensamientos y acciones
- Sin esto, el agente no tiene memoria y entra en bucles

### 4. Guardrails (Esenciales)

1. **Max Iterations**: Previene bucles infinitos
2. **Tool Whitelisting**: Previene ejecución de código peligroso
3. **Parser Check**: Previene errores por formato incorrecto

### 5. La Verdad sobre los Agentes

Un agente es simplemente:
- Un bucle `while`
- Parsing de strings
- Ejecución condicional de funciones
- Un string que crece (scratchpad)

Las librerías solo abstraen esto, pero la mecánica es la misma.

## Próximos Pasos

Ahora que entiendes los fundamentos, en el siguiente notebook (`05_agent_implementation.ipynb`) verás cómo LangChain abstrae todo esto con clases y métodos más elegantes. Pero ahora **sabes qué está pasando bajo el capó**.
